# ContaiNED: type-constrained zero-shot named entity disambiguation

Thesis Title: ***Named Entity Disambiguation using BERT***

Theodore Masi, Rhodes University

supervised by Marc Marais

Named Entity Disambiguation links an ambiguous mention in text to a specific
knowledge base entry. This pipeline groups co-referring mentions within a
document, pools their surrounding contexts, and resolves each group against
candidates drawn from the ZELDA alias table. No model component is trained.
Both the scoring encoder and the NER tagger are frozen pre-trained checkpoints,
so the system requires no annotated data in the target domain.

## Pipeline

```
document → NER tagging → mention clustering → candidate generation
         → context pooling → BERT scoring → hybrid ranking → predicted entity
```

The ordering carries the design. Clustering precedes candidate generation, so
one candidate set is retrieved and one scoring decision made per cluster
rather than per mention. Tagging precedes clustering, so a mention's predicted
type is available to constrain which mentions may be grouped together.

Candidates are ranked by a linear combination of three signals whose failure
modes are largely independent:

$$S_{final} = \alpha \cdot S_{BERT} + \beta \cdot S_{string} + \gamma \cdot S_{prior}$$

## The two configurations

The ablation compares two configurations that differ in **one condition only**:

| | Clustering rule |
|---|---|
| **Baseline** | merge when core-name containment **or** Levenshtein ratio ≥ 0.85 |
| **Container class** | the same, **plus** predicted types must agree — checked *before* any surface-form comparison, so string similarity cannot override it |

Everything else, including candidate generation, context pooling, encoder, scoring
function, weights, and cache policy is held identical, so the comparison isolates
the clustering strategy.

## Inputs

Two Kaggle datasets, with paths set in the configuration cell:

- **ZELDA test splits** as JSONL — nine splits, 4,816 documents, 27,277 annotated mentions
- **ZELDA alias-table pickles** — `zelda_mention_entities_counter.pickle` and `zelda_ids_to_titles.pickle`

Internet access must be enabled: candidate descriptions are fetched from the
Wikipedia API. Responses are cached to disk, so repeated runs over the same
documents issue no network requests.

## Running

Cells are ordered for top-to-bottom execution. Each configuration writes its own 
cache file, so candidate generation is warm and identical between them.

## Outputs

Written to `/kaggle/working/`:

| File | Contents |
|---|---|
| `results_{config}.csv` | per-split accuracy, candidate recall, conditional accuracy |
| `summary_{config}.json` | aggregate metrics, operation counters, configuration |
| `outcomes_{config}.json` | per-mention correctness, for paired significance testing |
| `table_comparison.csv` | side-by-side comparison of both configurations |
| `per_split_comparison.csv` | per-split deltas |
| `mcnemar.json` | discordant counts *b* and *c*, net difference, exact *p*-value |
| `misc_rates.csv` | predicted type distribution per split |


## Reproducibility

The pipeline is deterministic given a fixed cache, so accuracy figures and
operation counters reproduce exactly. Execution times carry unquantified
variance from contention on the shared Kaggle environment. The `User-Agent`
string in the Wikipedia session should carry a working contact address before
running at volume.

In [1]:
!pip install -q Levenshtein requests transformers psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.2 MB/s eta 0:00:00:00:01


In [2]:
import hashlib
import json
import os
import pickle
import random
import re
import time
import unicodedata
from collections import Counter

import Levenshtein
import numpy as np
import pandas as pd
import requests
import torch
import transformers
from scipy.stats import binomtest
from transformers import AutoTokenizer, AutoModel, pipeline

# record the execution environment for reproducibility
import platform, sys, psutil, subprocess

In [3]:
print("=== gpu ===")
print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True).stdout)
print("cuda version (torch build):", torch.version.cuda)

print("=== cpu / ram ===")
print(f"cores: {psutil.cpu_count(logical=False)} physical / {psutil.cpu_count(logical=True)} logical")
print(f"ram: {psutil.virtual_memory().total / 1e9:.1f} gb")

print("=== os / python ===")
print(platform.platform())
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

=== gpu ===
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04

cuda version (torch build): 12.8
=== cpu / ram ===
cores: 2 physical / 4 logical
ram: 33.7 gb
=== os / python ===
Linux-6.18.48+-x86_64-with-glibc2.35
python: 3.12.13
torch: 2.10.0+cu128
transformers: 5.0.0


In [4]:
# dataset paths
ZELDA_DIR = "/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl"
CANDIDATE_PICKLE = "/kaggle/input/datasets/theomasi/zeldapicks/zelda_mention_entities_counter.pickle"
IDS_PICKLE = "/kaggle/input/datasets/theomasi/zeldapicks/zelda_ids_to_titles.pickle"

SPLITS = ["aida-b", "cweb", "reddit-comments", "reddit-posts",
          "shadowlinks-shadow", "shadowlinks-tail", "shadowlinks-top",
          "tweeki", "wned-wiki"]

all_paths = [f"{ZELDA_DIR}/test_{s}.jsonl" for s in SPLITS]

# scoring encoder 
BERT_MODEL_NAME = "sentence-transformers/bert-base-nli-mean-tokens"
POOLING = "mean"              # 
BERT_MAX_LENGTH = 128         # truncation length for context and candidate text

# scoring weights: semantic, string, prior
ALPHA, BETA, GAMMA = 0.40, 0.35, 0.25

TOP_K = 10                    # candidates retrieved per alias
MAX_ALIASES = 5               # distinct aliases contributing per cluster
CONTEXT_WINDOW_CHARS = 200    # characters either side of a mention span
CLUSTER_THRESHOLD = 0.85      # minimum levenshtein ratio to merge two mentions
NER_CHUNK_CHARS = 3000        # document chunk size for the tagger

In [5]:
def load_zelda_jsonl(path):
    """read one zelda split into {id, text, mentions} records"""
    
    docs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            d = json.loads(line)
            mentions = [
                (s, e, title, wid)
                for (s, e), wid, title in zip(d["index"], d["wikipedia_ids"], d["wikipedia_titles"])
            ]
            # train and test splits name the document identifier differently
            doc_id = d.get("id", d.get("page_id"))
            docs.append({"id": doc_id, "text": d["text"], "mentions": mentions})
    return docs

In [6]:
class DiskCache:
    """persistent json cache for api responses and ner predictions"""

    def __init__(self, path):
        self.path = path
        if os.path.exists(path):
            with open(path, encoding="utf-8") as f:
                self.data = json.load(f)
        else:
            self.data = {}

    def get(self, key):
        return self.data.get(key)

    def set(self, key, value):
        self.data[key] = value

    def save(self):
        with open(self.path, "w", encoding="utf-8") as f:
            json.dump(self.data, f)

In [7]:
def load_zelda_alias_table(candidate_pickle, ids_pickle):
    """build the alias table and title->id map from the zelda pickles"""
    
    with open(candidate_pickle, "rb") as f:
        mention_entities_counter = pickle.load(f)
    with open(ids_pickle, "rb") as f:
        ids_to_titles = pickle.load(f)

    title_to_id = {title: wid for wid, title in ids_to_titles.items()}

    alias_table = {}
    for mention, entity_counts in mention_entities_counter.items():
        total = sum(entity_counts.values())
        sorted_items = sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)
        alias_table[mention] = [(title, count / total) for title, count in sorted_items]

    return alias_table, title_to_id

In [8]:
alias_table, title_to_id = load_zelda_alias_table(CANDIDATE_PICKLE, IDS_PICKLE)
print(f"alias table: {len(alias_table):,} surface forms")

alias table: 5,783,815 surface forms


In [9]:
_ALIAS_LOWER_INDEX = None


def build_alias_index(alias_table):
    """build a lower-cased index once, so case-insensitive lookup is O(1)"""
    
    idx = {}
    for k, v in alias_table.items():
        idx.setdefault(k.lower(), v)
    globals()["_ALIAS_LOWER_INDEX"] = idx
    print(f"alias lower-index built: {len(idx):,} keys")
    return idx

build_alias_index(alias_table)

alias lower-index built: 5,190,706 keys


{'revelation 13': [('Revelation 13', 0.6666666666666666),
  ('The Beast (Revelation)', 0.3333333333333333)],
 'apocalypse 13': [('Revelation 13', 1.0)],
 'henry lyte (botanist)': [('Henry Lyte (botanist)', 1.0)],
 'channel fleet': [('Channel Fleet', 0.75), ('Western Squadron', 0.25)],
 'typhoon weapon station': [('Typhoon Weapon Station', 1.0)],
 'conservative synagogue adath israel of riverdale': [('Conservative Synagogue Adath Israel of Riverdale',
   1.0)],
 'looper (film)': [('Looper (film)', 1.0)],
 'lateef adegbite': [('Lateef Adegbite', 1.0)],
 'nao ono': [('Nao Ono', 1.0)],
 'joe venuti': [('Joe Venuti', 1.0)],
 'fructuosus of braga': [('Fructuosus of Braga', 1.0)],
 'fructuosus bracarensis': [('Fructuosus of Braga', 1.0)],
 'ecny awards': [('ECNY Awards', 1.0)],
 'chennai 600028 ii': [('Chennai 600028 II', 1.0)],
 'louise de montmorency': [('Louise de Montmorency', 1.0)],
 'susan baird': [('Susan Baird', 1.0)],
 '1999 rice owls football team': [('1999 Rice Owls football team',

In [45]:
WIKI_API = "https://en.wikipedia.org/w/api.php"
MIN_REQUEST_INTERVAL = 1.0    # seconds between api calls

# wikipedia blocks requests with a generic user-agent so replace with your email
_WIKI_SESSION = requests.Session()
_WIKI_SESSION.headers.update({
    "User-Agent": "ContaiNED/1.0 (research project; contact:<your@email.here>)"
})

_last_request_time = [0.0]


def _throttle():
    elapsed = time.monotonic() - _last_request_time[0]
    if elapsed < MIN_REQUEST_INTERVAL:
        time.sleep(MIN_REQUEST_INTERVAL - elapsed)
    _last_request_time[0] = time.monotonic()


def wiki_get(params, max_retries=6):
    """throttled get against the wikipedia api with retry and backoff"""
    
    for attempt in range(max_retries):
        _throttle()
        try:
            r = _WIKI_SESSION.get(WIKI_API, params=params, timeout=15)
        except requests.exceptions.RequestException as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"  [wiki_get] network error ({e}); retrying in {wait:.1f}s "
                  f"(attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
            continue

        if r.status_code == 429 or r.status_code >= 500:
            retry_after = r.headers.get("Retry-After")
            wait = float(retry_after) if retry_after else (2 ** attempt) + random.uniform(0, 1)
            print(f"  [wiki_get] http {r.status_code}; backing off {wait:.1f}s "
                  f"(attempt {attempt + 1}/{max_retries})")
            time.sleep(wait)
            continue

        r.raise_for_status()
        return r.json()

    print(f"  [wiki_get] giving up after {max_retries} retries for params={params}")
    return None

In [11]:
def get_descriptions_batch(titles, cache: DiskCache, max_chars=300, batch_size=20):
    """fetch intro extracts for many titles in one request. Mediawiki accepts up to 50 pipe-separated titles so 
    20 stays well under that"""
    
    result = {}
    to_fetch = []
    for t in titles:
        cached = cache.get(f"desc::{t}::{max_chars}")
        if cached is not None:
            result[t] = cached
        else:
            to_fetch.append(t)

    for i in range(0, len(to_fetch), batch_size):
        batch = to_fetch[i:i + batch_size]
        params = {
            "action": "query", "prop": "extracts", "exintro": True,
            "explaintext": True, "redirects": True,
            "titles": "|".join(batch), "format": "json",
        }
        data = wiki_get(params)
        if data is None:
            # the whole batch failed
            for orig in batch:
                result.setdefault(orig, "")
            continue

        query = data.get("query", {})
        pages = query.get("pages", {})
        extract_by_title = {
            page.get("title"): page.get("extract", "")[:max_chars]
            for page in pages.values()
        }

        # resolve original -> normalised -> redirected title chains
        resolved = {t: t for t in batch}
        for norm in query.get("normalized", []):
            for orig, cur in resolved.items():
                if cur == norm["from"]:
                    resolved[orig] = norm["to"]
        for redir in query.get("redirects", []):
            for orig, cur in resolved.items():
                if cur == redir["from"]:
                    resolved[orig] = redir["to"]

        for orig in batch:
            extract = extract_by_title.get(resolved[orig], "")
            result[orig] = extract
            cache.set(f"desc::{orig}::{max_chars}", extract)
            cache.set(f"redirect::{orig}", resolved[orig])

    return result

In [12]:
def get_resolved_title(title, cache: DiskCache):
    """return the resolved title if it has been fetched, else the input"""
    return cache.get(f"redirect::{title}") or title

In [13]:
def get_candidates(query, top_k, cache: DiskCache):
    """live wikipedia search, used only when the alias table has no entry"""
    cached = cache.get(f"cand::{query}::{top_k}")
    if cached is not None:
        return cached
    params = {"action": "query", "list": "search", "srsearch": query,
              "format": "json", "srlimit": top_k}
    data = wiki_get(params)
    hits = data.get("query", {}).get("search", []) if data else []
    result = [[h["title"], h["pageid"]] for h in hits]
    cache.set(f"cand::{query}::{top_k}", result)
    return result

In [14]:
HONORIFICS = {"Mr", "Mrs", "Miss", "Dr", "Lord", "Count", "Van", "Sir"}


def get_core_name(name: str) -> str:
    """strip a leading honorific, which varies the surface form"""
    words = name.split()
    if words and words[0] in HONORIFICS:
        words = words[1:]
    return " ".join(words).lower()

In [15]:
def cluster_mentions(mention_spans, threshold=CLUSTER_THRESHOLD):
    """baseline clustering: surface-form evidence only"""
    sorted_mentions = sorted(mention_spans, key=lambda x: len(x[0]), reverse=True)
    assigned, clusters = set(), []

    for mention, span in sorted_mentions:
        if (mention, span) in assigned:
            continue
        head_core = get_core_name(mention)
        cluster = [(mention, span)]
        assigned.add((mention, span))

        for cand_mention, cand_span in sorted_mentions:
            if (cand_mention, cand_span) in assigned:
                continue
            cand_core = get_core_name(cand_mention)
            is_substring = (
                (cand_core in head_core and len(cand_core) > 2)
                or (head_core in cand_core and len(head_core) > 2)
            )
            is_similar = Levenshtein.ratio(mention.lower(), cand_mention.lower()) >= threshold
            if is_substring or is_similar:
                cluster.append((cand_mention, cand_span))
                assigned.add((cand_mention, cand_span))

        clusters.append(cluster)
    return clusters

In [16]:
def cluster_mentions_type_aware(mention_spans, span_to_type, threshold=CLUSTER_THRESHOLD):
    """container class clustering: identical to the baseline plus a type guard"""
    sorted_mentions = sorted(mention_spans, key=lambda x: len(x[0]), reverse=True)
    assigned, clusters = set(), []

    for mention, span in sorted_mentions:
        if (mention, span) in assigned:
            continue
        head_core = get_core_name(mention)
        head_type = span_to_type.get((mention, span), "MISC")
        cluster = [(mention, span)]
        assigned.add((mention, span))

        for cand_mention, cand_span in sorted_mentions:
            if (cand_mention, cand_span) in assigned:
                continue

            # type guard: reject cross-type merges before any surface-form
            cand_type = span_to_type.get((cand_mention, cand_span), "MISC")
            if cand_type != head_type:
                continue

            cand_core = get_core_name(cand_mention)
            is_substring = (
                (cand_core in head_core and len(cand_core) > 2)
                or (head_core in cand_core and len(head_core) > 2)
            )
            is_similar = Levenshtein.ratio(mention.lower(), cand_mention.lower()) >= threshold
            if is_substring or is_similar:
                cluster.append((cand_mention, cand_span))
                assigned.add((cand_mention, cand_span))

        clusters.append(cluster)
    return clusters

In [17]:
_ner_pipeline = None
_ner_timing_total = 0.0


def get_ner_tagger():
    global _ner_pipeline
    if _ner_pipeline is None:
        _ner_pipeline = pipeline(
            "ner",
            model="dslim/bert-base-NER",
            aggregation_strategy="simple",
            device=0 if torch.cuda.is_available() else -1,
        )
    return _ner_pipeline

In [18]:
def _text_hash(text):
    return hashlib.sha1(text.encode("utf-8")).hexdigest()

In [19]:
def get_predicted_entities(doc_text, cache):
    """tag one document, cached on a hash of its text documents are chunked because the underlying 
    bert-base model accepts at most 512 tokens"""
    key = f"ner::{_text_hash(doc_text)}"
    cached = cache.get(key)
    if cached is not None:
        return cached

    ner_tagger = get_ner_tagger()
    offsets = list(range(0, len(doc_text), NER_CHUNK_CHARS))
    chunks = [doc_text[o:o + NER_CHUNK_CHARS] for o in offsets]

    predicted_entities = []
    if chunks:
        try:
            batched_results = ner_tagger(chunks, batch_size=8)
        except Exception:
            batched_results = [[] for _ in chunks]
        for offset, ents in zip(offsets, batched_results):
            for ent in ents:
                predicted_entities.append({
                    "start": offset + ent["start"],
                    "end": offset + ent["end"],
                    "type": ent["entity_group"],
                })
    cache.set(key, predicted_entities)
    return predicted_entities

In [20]:
def get_mention_types(doc_text, mention_spans, cache):
    """assign each gold span the type of the predicted entity overlapping it most"""
    global _ner_timing_total
    t0 = time.perf_counter()
    predicted_entities = get_predicted_entities(doc_text, cache)
    span_to_type = {}
    for mention_text, span in mention_spans:
        s, e = span
        best_overlap, best_type = 0, "MISC"
        for pe in predicted_entities:
            overlap = min(e, pe["end"]) - max(s, pe["start"])
            if overlap > best_overlap:
                best_overlap, best_type = overlap, pe["type"]
        span_to_type[(mention_text, span)] = best_type
    _ner_timing_total += time.perf_counter() - t0
    return span_to_type

In [21]:
def get_candidates_alias(mention_text, alias_table, title_to_id, top_k=TOP_K):
    """alias table lookup: exact -> case-insensitive -> then core name"""
    idx = _ALIAS_LOWER_INDEX
    cands = alias_table.get(mention_text) or idx.get(mention_text.lower())
    if not cands:
        core = get_core_name(mention_text)
        if core and core != mention_text:
            cands = alias_table.get(core) or idx.get(core.lower())
    if not cands:
        return []

    # filter on page id before slicing so top_k candidates are actually returned
    result = []
    for title, prior in cands:
        pid = title_to_id.get(title)
        if pid is not None:
            result.append([title, int(pid), float(prior)])
            if len(result) == top_k:
                break
    return result

In [22]:

def get_candidates_multi_v2(aliases, top_k, cache, alias_table, title_to_id,
                            max_aliases=MAX_ALIASES):
    """pool candidates across the longest distinct aliases in a cluster"""
    unique_aliases = sorted(set(aliases), key=len, reverse=True)[:max_aliases]
    seen_pageids = set()
    merged = []

    for alias in unique_aliases:
        cands = get_candidates_alias(alias, alias_table, title_to_id, top_k=top_k)

        # fallback for mentions absent from the alias table
        if not cands:
            cands = [[title, int(pageid), 0.0]
                     for title, pageid in get_candidates(alias, top_k, cache)]

        for title, pageid, prior in cands:
            if pageid not in seen_pageids:
                seen_pageids.add(pageid)
                merged.append([title, pageid, prior])

    return merged

In [23]:
def context_window_marked(doc_text, span, window_chars=CONTEXT_WINDOW_CHARS):
    """extract a character window around a mention, delimiting it with markers."""
    s, e = span
    start = max(0, s - window_chars)
    end = min(len(doc_text), e + window_chars)
    return f"{doc_text[start:s]} [ENT] {doc_text[s:e]} [/ENT] {doc_text[e:end]}"


In [24]:
_tokenizer = None
_bert_model = None
_loaded_model_name = None
_device = "cuda" if torch.cuda.is_available() else "cpu"


def get_bert():
    """load the scoring encoder once"""
    global _tokenizer, _bert_model, _loaded_model_name
    if _bert_model is None or _loaded_model_name != BERT_MODEL_NAME:
        _tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
        _bert_model = AutoModel.from_pretrained(BERT_MODEL_NAME)
        _bert_model.to(_device)
        _bert_model.eval()
        _loaded_model_name = BERT_MODEL_NAME
    return _tokenizer, _bert_model

In [25]:
def _mean_pool(token_embeddings, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()
    return (token_embeddings * mask).sum(1) / mask.sum(1).clamp(min=1e-9)


def _first_last_pool(token_embeddings, attention_mask):
    # first token is [CLS]; last real token is the final position where the
    # attention mask is 1
    first = token_embeddings[:, 0, :]
    lengths = attention_mask.sum(1) - 1
    batch_idx = torch.arange(token_embeddings.size(0), device=token_embeddings.device)
    last = token_embeddings[batch_idx, lengths, :]
    return torch.cat([first, last], dim=-1)

In [26]:
def bert_embed(texts, batch_size=16):
    """pooled, normalised embeddings, so a dot product equals cosine similarity"""
    tokenizer, model = get_bert()
    all_embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True,
                            max_length=BERT_MAX_LENGTH, return_tensors="pt").to(_device)
            out = model(**enc)
            token_embeddings = out.last_hidden_state

            if POOLING == "first_last":
                pooled = _first_last_pool(token_embeddings, enc["attention_mask"])
            else:
                pooled = _mean_pool(token_embeddings, enc["attention_mask"])

            normed = torch.nn.functional.normalize(pooled, p=2, dim=1)
            all_embs.append(normed.cpu().numpy())
    return np.concatenate(all_embs, axis=0)

In [27]:
def compute_string_score(query, title):
    """levenshtein ratio plus a word-boundary substring bonus"""
    query = query.lower().strip()
    title = title.lower().strip()

    if query == title:
        return 1.0

    lev = Levenshtein.ratio(query, title)

    # whole-word matches only
    substr_bonus = 0.0
    if len(query) >= 4:
        pattern = r"\b" + re.escape(query) + r"\b"
        if re.search(pattern, title):
            substr_bonus = 0.12 * min(1.0, len(query) / len(title))
        elif re.search(pattern, query):
            substr_bonus = 0.12 * min(1.0, len(title) / len(query))

    return min(1.0, lev + substr_bonus)

In [28]:
def score_candidates_smart(context_text, candidates, cache, alias_query="", alpha=ALPHA, beta=BETA, gamma=GAMMA):
    """rank candidates by hybrid scoring formula"""
    if not candidates:
        return None, None, 0.0, []

    titles = [c[0] for c in candidates]
    pageids = [c[1] for c in candidates]
    priors = np.array([c[2] if len(c) > 2 else 0.0 for c in candidates])

    # candidates are represented by their title plus their intro extract
    desc_map = get_descriptions_batch(titles, cache)
    cand_texts = [f"{t}. {desc_map.get(t, '')}" for t in titles]

    embs = bert_embed([context_text] + cand_texts)
    ctx_emb, cand_embs = embs[0], embs[1:]
    sims = cand_embs @ ctx_emb

    string_sims = np.array([compute_string_score(alias_query.lower().strip(), t)
                            for t in titles])

    # alias distributions are heavily skewed, so the prior is log-transformed then min-max normalised 
    # onto the same scale as the other two signals
    if priors.max() > 0:
        priors = np.log(priors + 1e-6)
        priors = (priors - priors.min()) / (priors.max() - priors.min() + 1e-9)
    else:
        priors = np.zeros_like(priors)

    combined = alpha * sims + beta * string_sims + gamma * priors
    order = np.argsort(-combined)
    ranked = [(titles[i], pageids[i], float(combined[i]), float(sims[i])) for i in order]
    best_title, best_pageid, best_score, _ = ranked[0]

    return best_title, best_pageid, best_score, ranked

In [29]:
def disambiguate_document_base(doc_text, mention_spans, cache, alias_table, title_to_id,
                               top_k=TOP_K, max_aliases=MAX_ALIASES,
                               alpha=ALPHA, beta=BETA, gamma=GAMMA,
                               use_type_guard=True):
    """resolve every mention in one document"""
    predictions = {}

    if use_type_guard:
        span_to_type = get_mention_types(doc_text, mention_spans, cache)
        clusters = cluster_mentions_type_aware(mention_spans, span_to_type)
    else:
        clusters = cluster_mentions(mention_spans)

    for cluster in clusters:
        # pool the context windows of every mention in the cluster
        pooled_context = "".join(
            context_window_marked(doc_text, span) for _, span in cluster
        )
        aliases_in_cluster = [m for m, _ in cluster]

        candidates = get_candidates_multi_v2(
            aliases_in_cluster, top_k, cache, alias_table, title_to_id,
            max_aliases=max_aliases
        )

        if not candidates:
            # no candidate retrieved; recorded as a recall failure
            for m, span in cluster:
                predictions[(m, span)] = (None, None, 0.0, candidates, None)
            continue

        # the longest alias acts as the query for string similarity
        alias_query = sorted(aliases_in_cluster, key=len, reverse=True)[0]
        title, pageid, score, ranked = score_candidates_smart(
            pooled_context, candidates, cache, alias_query=alias_query,
            alpha=alpha, beta=beta, gamma=gamma
        )

        for m, span in cluster:
            predictions[(m, span)] = (title, pageid, score, candidates, ranked)

    return predictions

In [30]:
_latency_records = []
_current_config_label = None


def disambiguate_document_v2(doc_text, mention_spans, cache, alias_table, title_to_id,
                             top_k=TOP_K, max_aliases=MAX_ALIASES,
                             alpha=ALPHA, beta=BETA, gamma=GAMMA,
                             use_type_guard=True):
    """timing wrapper around disambiguate_document_base"""
    t0 = time.perf_counter()
    preds = disambiguate_document_base(
        doc_text, mention_spans, cache, alias_table, title_to_id,
        top_k=top_k, max_aliases=max_aliases,
        alpha=alpha, beta=beta, gamma=gamma, use_type_guard=use_type_guard,
    )
    elapsed = time.perf_counter() - t0

    n_mentions = len(mention_spans)
    if n_mentions > 0:
        _latency_records.append({
            "config": _current_config_label,
            "n_mentions": n_mentions,
            "doc_latency_s": elapsed,
            "per_mention_latency_s": elapsed / n_mentions,
        })
    return preds

In [31]:
def normalize_title(t):
    """strips parentheticals"""
    if not t:
        return ""
    t = unicodedata.normalize("NFC", t)
    t = t.strip().lower()
    t = re.sub(r"\s+", "_", t)
    t = re.sub(r"[_\-]+", "_", t)
    t = re.sub(r"\(.*?\)", "", t)
    t = re.sub(r"[^\w]", "", t)
    return t.strip("_")


def normalize_title_strict(t):
    """preserves the parenthetical disambiguator"""
    if not t:
        return ""
    t = unicodedata.normalize("NFC", t)
    t = t.strip().lower()
    t = re.sub(r"\s+", "_", t)
    t = re.sub(r"[_\-]+", "_", t)
    t = re.sub(r"[^\w()]", "", t)
    return t.strip("_")

In [32]:
def gold_match(pred_title, pred_pageid, gold_title, gold_id, pred_resolved_title=None, allow_substring=False):
    """strict matching"""
    if pred_pageid is not None and str(pred_pageid) == str(gold_id):
        return True
    ng = normalize_title_strict(gold_title)
    for t in (pred_title, pred_resolved_title):
        if t and normalize_title_strict(t) == ng:
            return True
    if allow_substring and pred_title and gold_title:
        np_, ng_ = normalize_title_strict(pred_title), normalize_title_strict(gold_title)
        if (np_ in ng_ or ng_ in np_) and 0.8 <= len(np_) / max(len(ng_), 1) <= 1.25:
            return True
    return False

In [33]:
def gold_match_lenient(pred_title, pred_pageid, gold_title, gold_id,pred_resolved_title=None):
    """the superseded rule, retained to measure how much accuracy it inflates"""
    if pred_pageid is not None and str(pred_pageid) == str(gold_id):
        return True
    norm_gold = normalize_title(gold_title)
    for t in [pred_title, pred_resolved_title]:
        if t is not None and normalize_title(t) == norm_gold:
            return True
    if pred_title and gold_title:
        npred, ngold = normalize_title(pred_title), normalize_title(gold_title)
        if npred in ngold or ngold in npred:
            ratio = len(npred) / len(ngold) if len(ngold) > 0 else 0
            if 0.5 <= ratio <= 2.0:
                return True
    return False

In [34]:
def evaluate_zelda_split_v2(jsonl_path, cache_path, alias_table, title_to_id,
                            limit_docs=None, top_k=TOP_K, save_every=5,
                            debug_samples=0, max_aliases=MAX_ALIASES,
                            alpha=ALPHA, beta=BETA, gamma=GAMMA, sleep=0.0,
                            use_type_guard=True):
    """evaluate a single split"""
    docs = load_zelda_jsonl(jsonl_path)
    if limit_docs is not None:
        docs = docs[:limit_docs]

    cache = DiskCache(cache_path)
    correct, total = 0, 0
    candidate_hits, candidate_total = 0, 0
    debug_printed = 0

    for doc_i, doc in enumerate(docs):
        try:
            text = doc["text"]
            mention_spans = [(text[s:e], (s, e)) for (s, e, _, _) in doc["mentions"]]

            preds = disambiguate_document_v2(
                text, mention_spans, cache, alias_table, title_to_id,
                top_k=top_k, max_aliases=max_aliases,
                alpha=alpha, beta=beta, gamma=gamma,
                use_type_guard=use_type_guard
            )

            for (s, e, gold_title, gold_id) in doc["mentions"]:
                mention_text = text[s:e]
                total += 1
                pred = preds.get((mention_text, (s, e)))
                if pred is None:
                    continue

                pred_title, pred_pageid, score, candidates, ranked = pred

                # candidate recall is measured on titles, before ranking
                gold_in_pool = any(
                    normalize_title(c_title) == normalize_title(gold_title)
                    or normalize_title(get_resolved_title(c_title, cache)) == normalize_title(gold_title)
                    for c_title, _, _ in candidates
                )
                if gold_in_pool:
                    candidate_hits += 1
                candidate_total += 1

                pred_resolved = get_resolved_title(pred_title, cache) if pred_title else None
                is_correct = gold_match(pred_title, pred_pageid, gold_title, gold_id,
                                        pred_resolved_title=pred_resolved)
                if is_correct:
                    correct += 1

                if gold_in_pool and debug_printed < debug_samples and ranked is not None:
                    debug_printed += 1
                    print(f"\n--- debug sample {debug_printed}/{debug_samples} ---")
                    print(f"mention: {mention_text!r}")
                    print(f"gold: {gold_title}")
                    print(f"correct: {is_correct}")
                    for c_title, c_pageid, c_score, c_bert in ranked:
                        marker = "<-- gold" if normalize_title(c_title) == normalize_title(gold_title) else ""
                        winner = "[picked]" if c_title == pred_title else ""
                        print(f"  {c_score:.4f} [{c_bert:.4f}] {c_title} {winner} {marker}")

        except Exception as e:
            print(f"[evaluate] skipping doc {doc.get('id')} after error: {e}")
            import traceback
            traceback.print_exc()

        if (doc_i + 1) % save_every == 0:
            cache.save()
            running_acc = correct / total if total else 0.0
            print(f"...{doc_i + 1}/{len(docs)} docs, running accuracy so far: {running_acc:.4f}")

        if sleep:
            time.sleep(sleep)

    cache.save()
    accuracy = correct / total if total else 0.0
    candidate_recall = candidate_hits / candidate_total if candidate_total else 0.0
    conditional_accuracy = correct / candidate_hits if candidate_hits else 0.0

    print(f"\n{'=' * 60}")
    print(jsonl_path)
    print(f"docs evaluated: {len(docs)}")
    print(f"mentions evaluated: {total}")
    print(f"candidate recall: {candidate_hits}/{candidate_total} = {candidate_recall:.4f}")
    print(f"accuracy: {correct}/{total} = {accuracy:.4f}")
    print(f"conditional accuracy: {correct}/{candidate_hits} = {conditional_accuracy:.4f}")
    print(f"{'=' * 60}")

    return {
        "accuracy": accuracy, "candidate_recall": candidate_recall,
        "conditional_accuracy": conditional_accuracy, "correct": correct,
        "total": total, "candidate_hits": candidate_hits,
        "candidate_total": candidate_total, "docs_evaluated": len(docs),
    }

In [35]:
def evaluate_all_zelda_splits(paths, cache_path, alias_table, title_to_id,
                              limit_docs=None, top_k=TOP_K, save_every=5,
                              max_aliases=MAX_ALIASES,
                              alpha=ALPHA, beta=BETA, gamma=GAMMA,
                              sleep=0.0, verbose=True, use_type_guard=True):
    """evaluate every split and aggregate"""
    all_results = []
    micro_correct = micro_total = 0
    micro_cand_hits = micro_cand_total = 0
    start_time = time.time()

    for i, path in enumerate(paths, 1):
        split_name = path.split("/")[-1].replace("test_", "").replace(".jsonl", "")
        if verbose:
            print(f"\n{'=' * 60}")
            print(f"[{i}/{len(paths)}] evaluating: {split_name}")
            print(f"{'=' * 60}")

        t0 = time.time()
        res = evaluate_zelda_split_v2(
            path, cache_path, alias_table, title_to_id,
            limit_docs=limit_docs, top_k=top_k, save_every=save_every,
            debug_samples=0, max_aliases=max_aliases,
            alpha=alpha, beta=beta, gamma=gamma, sleep=sleep,
            use_type_guard=use_type_guard,
        )
        elapsed = time.time() - t0

        all_results.append({
            "split": split_name, "docs": res["docs_evaluated"],
            "mentions": res["total"], "correct": res["correct"],
            "candidate_hits": res["candidate_hits"],
            "candidate_total": res["candidate_total"],
            "accuracy": res["accuracy"],
            "candidate_recall": res["candidate_recall"],
            "conditional_accuracy": res["conditional_accuracy"],
            "time_sec": elapsed,
        })

        micro_correct += res["correct"]
        micro_total += res["total"]
        micro_cand_hits += res["candidate_hits"]
        micro_cand_total += res["candidate_total"]

    total_time = time.time() - start_time
    df = pd.DataFrame(all_results)

    macro_metrics = {
        "macro_accuracy": df["accuracy"].mean(),
        "macro_candidate_recall": df["candidate_recall"].mean(),
        "macro_conditional_accuracy": df["conditional_accuracy"].mean(),
    }
    micro_metrics = {
        "micro_accuracy": micro_correct / micro_total if micro_total else 0.0,
        "micro_candidate_recall": micro_cand_hits / micro_cand_total if micro_cand_total else 0.0,
        "micro_conditional_accuracy": micro_correct / micro_cand_hits if micro_cand_hits else 0.0,
    }

    if verbose:
        print(f"\n{'=' * 60}")
        print("per-split results")
        print(f"{'=' * 60}")
        display_df = df.copy()
        for col in ["accuracy", "candidate_recall", "conditional_accuracy"]:
            display_df[col] = display_df[col].apply(lambda x: f"{x:.4f}")
        print(display_df.to_string(index=False))

        print(f"\n{'=' * 60}")
        print("aggregate metrics")
        print(f"{'=' * 60}")
        print(f"{'metric':<30} {'macro (avg per split)':>22} {'micro (pooled)':>20}")
        print("-" * 74)
        print(f"{'accuracy':<30} {macro_metrics['macro_accuracy']:>22.4f} {micro_metrics['micro_accuracy']:>20.4f}")
        print(f"{'candidate recall':<30} {macro_metrics['macro_candidate_recall']:>22.4f} {micro_metrics['micro_candidate_recall']:>20.4f}")
        print(f"{'conditional accuracy':<30} {macro_metrics['macro_conditional_accuracy']:>22.4f} {micro_metrics['micro_conditional_accuracy']:>20.4f}")
        print(f"\ntotal mentions evaluated: {micro_total:,}")
        print(f"total correct: {micro_correct:,}")
        print(f"total elapsed time: {total_time / 60:.1f} minutes")

    return df, macro_metrics, micro_metrics

In [36]:
# guarded so re-running the cell cannot stack wrappers on top of each other
if not globals().get("_INSTRUMENTED"):
    _INSTRUMENTED = True
    _cfg = None
    _counters = {}
    _outcomes = {}

    def reset_counters(label):
        global _cfg
        _cfg = label
        _counters.clear()
        _counters.update({"clusters_total": 0, "clusters_scored": 0,
                          "candidates": 0, "bert_seqs": 0,
                          "desc_titles": 0, "wiki_calls": 0})
        _outcomes[label] = []

    _o_score = score_candidates_smart
    def score_candidates_smart(context_text, candidates, cache, **kw):
        _counters["clusters_scored"] += 1
        _counters["candidates"] += len(candidates)
        _counters["bert_seqs"] += 1 + len(candidates)
        return _o_score(context_text, candidates, cache, **kw)

    _o_cm, _o_cmt = cluster_mentions, cluster_mentions_type_aware
    def cluster_mentions(*a, **kw):
        c = _o_cm(*a, **kw); _counters["clusters_total"] += len(c); return c
    def cluster_mentions_type_aware(*a, **kw):
        c = _o_cmt(*a, **kw); _counters["clusters_total"] += len(c); return c

    _o_desc = get_descriptions_batch
    def get_descriptions_batch(titles, cache, **kw):
        _counters["desc_titles"] += len(titles)
        return _o_desc(titles, cache, **kw)

    _o_wiki = wiki_get
    def wiki_get(params, **kw):
        _counters["wiki_calls"] += 1
        return _o_wiki(params, **kw)

    _o_gm = gold_match
    def gold_match(*a, **kw):
        r = _o_gm(*a, **kw)
        _outcomes[_cfg].append((str(a[3]), bool(r)))
        return r

    reset_counters("init")
    print("instrumented")

instrumented


In [37]:
# experiment 1.1: how much accuracy the lenient matching rule inflates
_keep_strict = gold_match

_, _, micro_strict = evaluate_all_zelda_splits(
    paths=all_paths, cache_path="wiki_cache_smoke.json",
    alias_table=alias_table, title_to_id=title_to_id,
    limit_docs=5, alpha=0.20, beta=0.45, gamma=0.35,
    use_type_guard=True, verbose=False)

gold_match = gold_match_lenient
_, _, micro_lenient = evaluate_all_zelda_splits(
    paths=all_paths, cache_path="wiki_cache_smoke.json",
    alias_table=alias_table, title_to_id=title_to_id,
    limit_docs=5, alpha=0.20, beta=0.45, gamma=0.35,
    use_type_guard=True, verbose=False)
gold_match = _keep_strict      # restore before anything else runs

print(f"lenient   : {micro_lenient['micro_accuracy']:.4f}")
print(f"strict    : {micro_strict['micro_accuracy']:.4f}")
print(f"inflation : {micro_lenient['micro_accuracy'] - micro_strict['micro_accuracy']:+.4f}")

config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

... (10 outputs left out)

BertModel LOAD REPORT from: sentence-transformers/bert-base-nli-mean-tokens
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...5/5 docs, running accuracy so far: 0.5616

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_aida-b.jsonl
docs evaluated: 5
mentions evaluated: 146
candidate recall: 107/146 = 0.7329
accuracy: 82/146 = 0.5616
conditional accuracy: 82/107 = 0.7664


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


...5/5 docs, running accuracy so far: 0.8654

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_cweb.jsonl
docs evaluated: 5
mentions evaluated: 156
candidate recall: 153/156 = 0.9808
accuracy: 135/156 = 0.8654
conditional accuracy: 135/153 = 0.8824
...5/5 docs, running accuracy so far: 0.8667

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_reddit-comments.jsonl
docs evaluated: 5
mentions evaluated: 15
candidate recall: 15/15 = 1.0000
accuracy: 13/15 = 0.8667
conditional accuracy: 13/15 = 0.8667
...5/5 docs, running accuracy so far: 0.5714

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_reddit-posts.jsonl
docs evaluated: 5
mentions evaluated: 7
candidate recall: 7/7 = 1.0000
accuracy: 4/7 = 0.5714
conditional accuracy: 4/7 = 0.5714
...5/5 docs, running accuracy so far: 0.2000

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_shadowlinks-shadow.jsonl
docs evaluated: 5
mentions evaluated: 5
candidate recall: 2/5 = 0.4000
accuracy

In [38]:
# experiment 1.2: sweep the semantic weight for each candidate encoder 
for model_name, pooling in [("bert-base-uncased", "first_last"),
                            ("sentence-transformers/bert-base-nli-mean-tokens", "mean")]:
    BERT_MODEL_NAME, POOLING = model_name, pooling
    _bert_model = None          # force get_bert() to reload
    _loaded_model_name = None

    for a, b, g in [(0.00, 0.50, 0.50), (0.20, 0.45, 0.35),
                    (0.40, 0.35, 0.25), (0.70, 0.20, 0.10)]:
        _, _, micro = evaluate_all_zelda_splits(
            paths=all_paths, cache_path="wiki_cache_smoke.json",
            alias_table=alias_table, title_to_id=title_to_id,
            limit_docs=5, alpha=a, beta=b, gamma=g,
            use_type_guard=True, verbose=False)
        print(f"{model_name:46} alpha={a:.2f}  acc={micro['micro_accuracy']:.4f}  "
              f"cond={micro['micro_conditional_accuracy']:.4f}")

# restore the selected configuration
BERT_MODEL_NAME = "sentence-transformers/bert-base-nli-mean-tokens"
POOLING = "mean"
_bert_model = None
_loaded_model_name = None

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...5/5 docs, running accuracy so far: 0.5616

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_aida-b.jsonl
docs evaluated: 5
mentions evaluated: 146
candidate recall: 107/146 = 0.7329
accuracy: 82/146 = 0.5616
conditional accuracy: 82/107 = 0.7664
...5/5 docs, running accuracy so far: 0.8590

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_cweb.jsonl
docs evaluated: 5
mentions evaluated: 156
candidate recall: 153/156 = 0.9808
accuracy: 134/156 = 0.8590
conditional accuracy: 134/153 = 0.8758
...5/5 docs, running accuracy so far: 0.8667

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_reddit-comments.jsonl
docs evaluated: 5
mentions evaluated: 15
candidate recall: 15/15 = 1.0000
accuracy: 13/15 = 0.8667
conditional accuracy: 13/15 = 0.8667
...5/5 docs, running accuracy so far: 0.5714

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_reddit-posts.jsonl
docs evaluated: 5
mentions evaluated: 7
candidate recall: 7/7 = 1.0000
accuracy

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/bert-base-nli-mean-tokens
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...5/5 docs, running accuracy so far: 0.5616

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_aida-b.jsonl
docs evaluated: 5
mentions evaluated: 146
candidate recall: 107/146 = 0.7329
accuracy: 82/146 = 0.5616
conditional accuracy: 82/107 = 0.7664
...5/5 docs, running accuracy so far: 0.8590

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_cweb.jsonl
docs evaluated: 5
mentions evaluated: 156
candidate recall: 153/156 = 0.9808
accuracy: 134/156 = 0.8590
conditional accuracy: 134/153 = 0.8758
...5/5 docs, running accuracy so far: 0.8667

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_reddit-comments.jsonl
docs evaluated: 5
mentions evaluated: 15
candidate recall: 15/15 = 1.0000
accuracy: 13/15 = 0.8667
conditional accuracy: 13/15 = 0.8667
...5/5 docs, running accuracy so far: 0.5714

/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_reddit-posts.jsonl
docs evaluated: 5
mentions evaluated: 7
candidate recall: 7/7 = 1.0000
accuracy

In [39]:
# smoke test: two documents, asserting the pipeline actually produces predictions
reset_counters("smoke")
_, _, m = evaluate_all_zelda_splits(
    paths=[all_paths[0]], cache_path="wiki_cache_smoke.json",
    alias_table=alias_table, title_to_id=title_to_id, limit_docs=2,
    verbose=False)

assert m["micro_accuracy"] > 0, "no predictions produced; do not start the full run"
assert _counters["clusters_scored"] > 0, "counters not firing"
print("ok:", m["micro_accuracy"], dict(_counters))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/bert-base-nli-mean-tokens
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



/kaggle/input/datasets/theomasi/zeldabert/test_data/jsonl/test_aida-b.jsonl
docs evaluated: 2
mentions evaluated: 65
candidate recall: 44/65 = 0.6769
accuracy: 26/65 = 0.4000
conditional accuracy: 26/44 = 0.5909
ok: 0.4 {'clusters_total': 44, 'clusters_scored': 44, 'candidates': 272, 'bert_seqs': 316, 'desc_titles': 272, 'wiki_calls': 0}


In [40]:
def save_config_results(label, df, macro, micro, counters):
    """write per-split results, summary metrics and per-mention outcomes"""
    df.to_csv(f"/kaggle/working/results_{label}.csv", index=False)
    with open(f"/kaggle/working/summary_{label}.json", "w") as f:
        json.dump({"config": label, "macro": macro, "micro": micro,
                   "counters": counters, "model": BERT_MODEL_NAME,
                   "pooling": POOLING,
                   "alpha": ALPHA, "beta": BETA, "gamma": GAMMA}, f, indent=2)
    with open(f"/kaggle/working/outcomes_{label}.json", "w") as f:
        json.dump(_outcomes[label], f)
    print(f"saved {label}")

In [41]:
COMMON = dict(
    paths=all_paths, alias_table=alias_table, title_to_id=title_to_id,
    alpha=ALPHA, beta=BETA, gamma=GAMMA, save_every=10, verbose=True,
)

reset_counters("baseline")
baseline_df, baseline_macro, baseline_micro = evaluate_all_zelda_splits(
    cache_path="wiki_cache_baseline.json", use_type_guard=False, **COMMON)
baseline_counters = dict(_counters)      # snapshot before the next reset clears it
print("baseline:", baseline_counters)
save_config_results("baseline", baseline_df, baseline_macro, baseline_micro, baseline_counters)

reset_counters("typeguard")
typeguard_df, typeguard_macro, typeguard_micro = evaluate_all_zelda_splits(
    cache_path="wiki_cache_typeguard.json", use_type_guard=True, **COMMON)
typeguard_counters = dict(_counters)
print("typeguard:", typeguard_counters)
save_config_results("typeguard", typeguard_df, typeguard_macro, typeguard_micro, typeguard_counters)


[1/9] evaluating: aida-b
...10/231 docs, running accuracy so far: 0.6357
...20/231 docs, running accuracy so far: 0.6404
...30/231 docs, running accuracy so far: 0.5876
...40/231 docs, running accuracy so far: 0.6141
...50/231 docs, running accuracy so far: 0.6313
...60/231 docs, running accuracy so far: 0.6424
...70/231 docs, running accuracy so far: 0.6434
...80/231 docs, running accuracy so far: 0.6498
...90/231 docs, running accuracy so far: 0.6603
...100/231 docs, running accuracy so far: 0.6637
...110/231 docs, running accuracy so far: 0.6695
...120/231 docs, running accuracy so far: 0.6744
...130/231 docs, running accuracy so far: 0.6769
...140/231 docs, running accuracy so far: 0.6816
...150/231 docs, running accuracy so far: 0.6802
...160/231 docs, running accuracy so far: 0.6787
...170/231 docs, running accuracy so far: 0.6796
...180/231 docs, running accuracy so far: 0.6824
...190/231 docs, running accuracy so far: 0.7051
...200/231 docs, running accuracy so far: 0.7160
...

In [42]:
comp = pd.DataFrame({
    "Baseline": [baseline_micro["micro_accuracy"], baseline_micro["micro_candidate_recall"],
                 baseline_micro["micro_conditional_accuracy"],
                 baseline_counters["clusters_total"], baseline_counters["clusters_scored"],
                 baseline_counters["candidates"], baseline_counters["bert_seqs"],
                 baseline_counters["desc_titles"], baseline_counters["wiki_calls"]],
    "Type-guarded": [typeguard_micro["micro_accuracy"], typeguard_micro["micro_candidate_recall"],
                     typeguard_micro["micro_conditional_accuracy"],
                     typeguard_counters["clusters_total"], typeguard_counters["clusters_scored"],
                     typeguard_counters["candidates"], typeguard_counters["bert_seqs"],
                     typeguard_counters["desc_titles"], typeguard_counters["wiki_calls"]],
}, index=["Micro accuracy", "Micro candidate recall", "Micro conditional accuracy",
          "Clusters formed", "Clusters scored", "Candidates generated",
          "BERT sequences", "Description titles", "Wikipedia API calls"])
comp["Delta"] = comp["Type-guarded"] - comp["Baseline"]
comp["Pct change"] = (comp["Delta"] / comp["Baseline"] * 100).round(2)
comp.to_csv("/kaggle/working/table_comparison.csv")
print(comp.to_string())

merged = baseline_df.merge(typeguard_df, on="split", suffixes=("_base", "_tg"))
merged["acc_delta"] = merged.accuracy_tg - merged.accuracy_base
merged["recall_delta"] = merged.candidate_recall_tg - merged.candidate_recall_base
merged.to_csv("/kaggle/working/per_split_comparison.csv", index=False)

                                 Baseline   Type-guarded        Delta  Pct change
Micro accuracy                   0.663636       0.667339     0.003703        0.56
Micro candidate recall           0.908054       0.898559    -0.009495       -1.05
Micro conditional accuracy       0.730833       0.742676     0.011844        1.62
Clusters formed              20449.000000   21689.000000  1240.000000        6.06
Clusters scored              20442.000000   21682.000000  1240.000000        6.07
Candidates generated        120933.000000  126954.000000  6021.000000        4.98
BERT sequences              141375.000000  148636.000000  7261.000000        5.14
Description titles          120933.000000  126954.000000  6021.000000        4.98
Wikipedia API calls          13125.000000   13285.000000   160.000000        1.22


In [43]:
# mcnemar's test on the paired per-mention outcomes 
A, B = _outcomes["baseline"], _outcomes["typeguard"]
if [x[0] for x in A] == [x[0] for x in B]:
    b = sum(1 for x, y in zip(A, B) if x[1] and not y[1])
    c = sum(1 for x, y in zip(A, B) if y[1] and not x[1])
    res = {"b": b, "c": c, "net": c - b, "discordant": b + c,
           "p_value": binomtest(c, b + c, 0.5).pvalue if b + c else 1.0}
else:
    res = {"error": f"order differs: {len(A)} vs {len(B)} outcomes"}

with open("/kaggle/working/mcnemar.json", "w") as f:
    json.dump(res, f, indent=2)
print(res)

{'b': 304, 'c': 405, 'net': 101, 'discordant': 709, 'p_value': np.float64(0.0001689614627855042)}


In [44]:
def misc_rate(paths, cache_path, limit_docs=None):
    """tally predicted type distribution per split from the cached ner output"""
    cache = DiskCache(cache_path)
    rows = []
    for p in paths:
        docs = load_zelda_jsonl(p)
        if limit_docs:
            docs = docs[:limit_docs]
        tally = Counter()
        for doc in docs:
            text = doc["text"]
            spans = [(text[s:e], (s, e)) for (s, e, _, _) in doc["mentions"]]
            for t in get_mention_types(text, spans, cache).values():
                tally[t] += 1
        n = sum(tally.values())
        rows.append({"split": p.split("/")[-1].replace("test_", "").replace(".jsonl", ""),
                     "mentions": n,
                     "MISC_rate": tally["MISC"] / max(n, 1),
                     **{k: tally[k] / max(n, 1) for k in ("PER", "ORG", "LOC")}})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df


misc_df = misc_rate(all_paths, "wiki_cache_typeguard.json")
misc_df.to_csv("/kaggle/working/misc_rates.csv", index=False)

             split  mentions  MISC_rate      PER      ORG      LOC
            aida-b      4485   0.202899 0.204682 0.282274 0.310145
              cweb     11116   0.504948 0.097337 0.187118 0.210597
   reddit-comments       638   0.705329 0.115987 0.068966 0.109718
      reddit-posts       704   0.565341 0.170455 0.105114 0.159091
shadowlinks-shadow       904   0.240044 0.317478 0.200221 0.242257
  shadowlinks-tail       901   0.260821 0.441731 0.185350 0.112098
   shadowlinks-top       904   0.243363 0.325221 0.171460 0.259956
            tweeki       860   0.194186 0.165116 0.260465 0.380233
         wned-wiki      6765   0.386844 0.095787 0.191574 0.325795
